<a href="https://colab.research.google.com/github/BardRimon/Study/blob/main/InformationExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задачи

5. Создать пайплайн с оценкой качества
6. Применить методы NER из прошлого курса (морфологические анализаторы и открытые библиотеки) и оценить их по показателям SemEval
7. Применить модели, обученные на FactRuEval (deeppavlov и др.), на размеченном корпусе с оценкой качества выделения сущностей, например, локация и организация
8. Сравнить качество существующих моделей и последних LLM

1. Скачать тестовую часть NEREL.
2. Применить LLM для извлечения вложенных именованных сущностей и отношений между ними в режиме 0- и 5-shot промптинга с оценкой качества.


## тестовый прогон

In [8]:
# @title 0. Cкачивание nerel
!git clone https://github.com/nerel-ds/NEREL.git

Cloning into 'NEREL'...
remote: Enumerating objects: 3078, done.
remote: Counting objects: 100% (3078/3078), done.
remote: Compressing objects: 100% (2896/2896), done.
remote: Total 3078 (delta 1159), reused 2072 (delta 179), pack-reused 0 (from 0)
Receiving objects: 100% (3078/3078), 3.23 MiB | 4.38 MiB/s, done.
Resolving deltas: 100% (1159/1159), done.


In [12]:
import os
import json
from collections import defaultdict

# 1. Настройки путей (проверьте, где лежит скачанная папка)
# Если вы сделали git clone, путь скорее всего такой:
NEREL_PATH = '/content/NEREL/NEREL-v1.1/train'  # Или ./nerel/train, проверьте имя папки через !ls
OUTPUT_FILE = 'dataset.json'

# 2. Маппинг (Словарь перевода тегов NEREL в ваши поля)
# В NEREL теги на английском (PERSON, ORGANIZATION), а ваш код ждет "Кто", "Стоимость"
TAG_MAPPING = {
    'PERSON': 'Кто',
    'ORGANIZATION': 'Кто',      # Организации тоже могут быть субъектами
    'MONEY': 'Стоимость',
    'DATE': 'Дата',
    'COUNTRY': 'Регион',
    'CITY': 'Регион',
    'STATE_OR_PROVINCE': 'Регион',
    # 'EVENT': 'Цель'           # Можно попробовать мапить события на Цель, но это не всегда точно
}

def parse_ann_file(ann_path, txt_content):
    """Парсит .ann файл и извлекает сущности по оффсетам."""
    entities = defaultdict(list)

    with open(ann_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if not parts[0].startswith('T'): continue # Нас интересуют только сущности (T-tags)

            # parts[1] выглядит как "PERSON 10 20"
            entity_info = parts[1].split()
            tag = entity_info[0]

            # Если этот тег нам интересен (есть в маппинге)
            if tag in TAG_MAPPING:
                # Извлекаем текст сущности (parts[2])
                entity_text = parts[2]

                # Записываем в наш словарь под нужным русским ключом
                my_key = TAG_MAPPING[tag]
                entities[my_key].append(entity_text)

    return entities

# 3. Основной цикл конвертации
converted_data = []

if not os.path.exists(NEREL_PATH):
    print(f"Ошибка: Папка {NEREL_PATH} не найдена. Проверьте путь (сделайте !ls)")
else:
    files = [f for f in os.listdir(NEREL_PATH) if f.endswith('.txt')]
    print(f"Найдено {len(files)} файлов. Начинаю обработку...")

    for txt_file in files:
        base_name = txt_file[:-4]
        txt_path = os.path.join(NEREL_PATH, txt_file)
        ann_path = os.path.join(NEREL_PATH, base_name + '.ann')

        if not os.path.exists(ann_path): continue

        # Читаем текст
        with open(txt_path, 'r', encoding='utf-8') as f:
            text = f.read()

        # Извлекаем сущности
        labels = parse_ann_file(ann_path, text)

        # Добавляем пустые списки для полей, которых не нашли (чтобы код не падал)
        for key in ["Кто", "Тип вложения", "Цель", "Регион", "Стоимость", "Дата"]:
            if key not in labels:
                labels[key] = []

        # Собираем элемент
        converted_data.append({
            "text": text,
            "labels": labels
        })

    # 4. Сохраняем результат
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(converted_data, f, ensure_ascii=False, indent=4)

    print(f"Готово! Создан файл {OUTPUT_FILE} с {len(converted_data)} примерами.")
    print("Теперь загрузите его в ячейке 2 вместо старого кода.")

Найдено 746 файлов. Начинаю обработку...
Готово! Создан файл dataset.json с 746 примерами.
Теперь загрузите его в ячейке 2 вместо старого кода.


In [9]:
# @title 1. Установка зависимостей
# Устанавливаем Natasha для правил, Transformers для нейросетей
!pip install -q natasha pymorphy2 transformers accelerate bitsandbytes scikit-learn
print("Установка завершена.")

Установка завершена.


In [ ]:
# @title 2. Подготовка данных (Обновленная)
import json
import re
from natasha import (
    Segmenter, MorphVocab,
    NewsEmbedding, NewsNERTagger,
    Doc
)
import pymorphy2
# ... импорты метрик ...

# ЗАГРУЗКА РЕАЛЬНОГО ДАТАСЕТА
try:
    with open('dataset.json', 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    print(f"Успешно загружен датасет NEREL: {len(raw_data)} документов.")
except FileNotFoundError:
    print("Файл dataset.json не найден! Сначала выполните скрипт конвертации.")
    # Тут можно оставить фоллбэк на игрушечные данные, если хотите

In [13]:


# Дальше функция calculate_metrics без изменений...

# --- 2. Метрики (Strict & Partial) ---
def calculate_metrics(pred_data, true_data):
    """Считает точность (Precision), полноту (Recall) и F1."""
    tp_strict, tp_partial, fp, fn = 0, 0, 0, 0

    for i in range(len(pred_data)):
        pred = pred_data[i]
        true = true_data[i]

        # Нормализация
        p_set = set([str(x).lower().strip() for x in pred if x])
        t_set = set([str(x).lower().strip() for x in true if x])

        if not t_set and not p_set: continue # Оба пустые - ок

        # Strict match (полное совпадение)
        if p_set == t_set:
            tp_strict += 1

        # Partial match (пересечение слов)
        # Если есть хотя бы одно общее слово (грубая оценка)
        elif any(word in ' '.join(t_set) for p_str in p_set for word in p_str.split()):
            tp_partial += 1
        elif p_set and not t_set:
            fp += 1
        elif t_set and not p_set:
            fn += 1
        else:
            # Если не совпало совсем
            fp += 1
            fn += 1

    # Расчет для Partial (считаем частичное как 0.5 успеха)
    total_relevant = len(pred_data) # Упрощенно для примера
    accuracy = (tp_strict + 0.5 * tp_partial) / (tp_strict + tp_partial + fp + fn + 1e-9)
    return round(accuracy, 2)

print("Данные загружены. Количество примеров:", len(raw_data))

Успешно загружен датасет NEREL: 746 документов.
Данные загружены. Количество примеров: 746


In [15]:
# @title 3. Метод Natasha (Rule-based)


class NatashaExtractor:
    def __init__(self):
        self.segmenter = Segmenter()
        self.emb = NewsEmbedding()
        self.ner_tagger = NewsNERTagger(self.emb)
        self.morph = pymorphy2.MorphAnalyzer()

    def extract(self, text):
        doc = Doc(text)
        doc.segment(self.segmenter)
        doc.tag_ner(self.ner_tagger)

        # 1. Извлечение Сумм (Regex)
        money_regex = r'(\d+(?:[\.,]\d+)?\s*(?:млрд|млн|тыс)?\s*(?:руб|доллар|евро)[а-я]*)'
        cost = re.findall(money_regex, text, re.IGNORECASE)

        # 2. Извлечение Дат (Regex + Natasha)
        date_regex = r'(\d{4}\s*год[а-у]?)'
        date = re.findall(date_regex, text, re.IGNORECASE)

        # 3. Извлечение Организаций и Локаций (Natasha NER)
        who = []
        region = []
        for span in doc.spans:
            if span.type == 'ORG':
                who.append(span.text)
            elif span.type == 'LOC':
                region.append(span.text)

        # 4. Тип вложения (Лемматизация глаголов)
        verbs = {'инвестировать', 'выделить', 'вложить', 'направить'}
        actions = []
        for token in doc.tokens:
            p = self.morph.parse(token.text)[0]
            if p.normal_form in verbs:
                actions.append(token.text)

        return {
            "Кто": who,
            "Тип вложения": actions,
            "Цель": [], # Сложно для правил
            "Регион": region,
            "Стоимость": cost,
            "Дата": date
        }

# Тест
extractor_natasha = NatashaExtractor()
res = extractor_natasha.extract(raw_data[0]['text'])
print("Natasha результат:", res)

Natasha результат: {'Кто': ['ФИДЕ', 'ФИДЕ', 'ФИДЕ'], 'Тип вложения': [], 'Цель': [], 'Регион': ['Элисте', 'Дортмунде'], 'Стоимость': [], 'Дата': ['1975 года', '1975 года', '2005 год', '2005 году']}


## попытка 2

### Пункт 5 pipline

In [26]:
import abc
from dataclasses import dataclass
from typing import List, Set, Dict, Tuple, Any, Optional
from collections import defaultdict
# Сторонние библиотеки
import torch
from transformers import pipeline
from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsNERTagger,
    NamesExtractor,
    DatesExtractor,
    MoneyExtractor,
    AddrExtractor,
    Doc
)

# ==========================================
# 1. Единый формат данных (Data Contract)
# ==========================================

@dataclass
class Entity:
    """
    Универсальное представление сущности.
    """
    text: str          # Текст сущности
    label: str         # Тип (PER, ORG, LOC, DATE, MONEY)
    start: int         # Начальный индекс в тексте
    end: int           # Конечный индекс
    confidence: Optional[float] = None  # Уверенность модели (если есть)

# ==========================================
# 2. Абстрактный базовый класс
# ==========================================

class BaseNERPipeline(abc.ABC):
    """
    Интерфейс для всех NER-экстракторов.
    """

    @abc.abstractmethod
    def extract(self, text: str) -> List[Entity]:
        """
        Основной метод извлечения сущностей.
        """
        pass

# ==========================================
# 3. Реализация Сценария А: Natasha (Rules + Embeddings)
# ==========================================

class NatashaPipeline(BaseNERPipeline):
    def __init__(self):
        # Инициализация компонентов Natasha
        # Это "ленивая" загрузка, выполняется один раз при старте
        self.segmenter = Segmenter()
        self.morph_vocab = MorphVocab()

        # Эмбеддинги для NER (names, orgs, locs)
        emb = NewsEmbedding()
        self.ner_tagger = NewsNERTagger(emb)

        # Экстракторы на правилах (очень точные для дат и денег)
        self.dates_extractor = DatesExtractor(self.morph_vocab)
        self.money_extractor = MoneyExtractor(self.morph_vocab)
        self.names_extractor = NamesExtractor(self.morph_vocab) # Для нормализации имен

    def extract(self, text: str) -> List[Entity]:
        doc = Doc(text)

        # 1. Сегментация (разбиение на токены и предложения)
        doc.segment(self.segmenter)

        # 2. NER через нейросеть (Slovnet внутри Natasha)
        doc.tag_ner(self.ner_tagger)

        # 3. Нормализация (приведение к начальной форме)
        for span in doc.spans:
            span.normalize(self.morph_vocab)

        entities = []

        # Конвертация спанов Natasha в наш формат Entity
        for span in doc.spans:
            entities.append(Entity(
                text=span.normal, # Или span.text для исходного
                label=span.type,  # PER, LOC, ORG
                start=span.start,
                end=span.stop,
                confidence=1.0    # Natasha не отдает confidence явно в простом API
            ))

        # 4. Дополнительное извлечение фактов (Правила)
        # Пример для Денег (Money)
        money_matches = self.money_extractor(text)
        for match in money_matches:
            entities.append(Entity(
                text=text[match.start:match.stop],
                label='MONEY',
                start=match.start,
                end=match.stop,
                confidence=1.0
            ))

        # Сортируем по появлению в тексте
        return sorted(entities, key=lambda x: x.start)

# ==========================================
# 4. Реализация Сценария B: Hugging Face (Transformers)
# ==========================================

class TransformersPipeline(BaseNERPipeline):
    def __init__(self, model_name: str = "Babelscape/wikineural-multilingual-ner", device: int = -1):
        """
        Args:
            model_name: Имя модели. По умолчанию используем проверенную Babelscape.
            device: -1 для CPU, 0 для GPU.
        """
        print(f"Loading transformer model: {model_name}...")
        try:
            self.pipeline = pipeline(
                "ner",
                model=model_name,
                tokenizer=model_name,
                aggregation_strategy="simple", # Склеивает токены (Влад + ##имир -> Владимир)
                device=device
            )
        except OSError as e:
            print(f"CRITICAL ERROR: Не удалось загрузить модель '{model_name}'.")
            print("Проверь подключение к интернету или правильность имени модели на https://huggingface.co/models")
            raise e

    def extract(self, text: str) -> List[Entity]:
        # Инференс
        try:
            raw_results = self.pipeline(text)
        except Exception as e:
            print(f"Error during inference: {e}")
            return []

        entities = []
        for item in raw_results:
            # У этой модели теги приходят в формате 'PER', 'LOC', 'ORG' (без B-/I-)
            # thanks to aggregation_strategy="simple"

            # Фильтруем мусор, если уверенность низкая (опционально)
            if item['score'] < 0.30:
                continue

            entities.append(Entity(
                text=item['word'],
                label=item['entity_group'], # Babelscape возвращает entity_group
                start=item['start'],
                end=item['end'],
                confidence=float(item['score'])
            ))

        # Сортировка по порядку появления в тексте
        return sorted(entities, key=lambda x: x.start)



### Тест pipline

In [25]:
# ==========================================
# 5. Демонстрация (Client Code)
# ==========================================

if __name__ == "__main__":
    text = "Иван Петров купил акции Газпрома в Москве за 10000 рублей 5 мая 2023 года."

    print(f"Исходный текст: {text}\n")

    # Сценарий A: Natasha
    print("--- Natasha Extraction ---")
    natasha_pipe = NatashaPipeline()
    for ent in natasha_pipe.extract(text):
        print(f"[{ent.label}] {ent.text} ({ent.start}-{ent.end})")

    print("\n--- Transformers Extraction (ruBert-tiny2) ---")
    # Используем CPU для демо
    hf_pipe = TransformersPipeline(device=-1)
    for ent in hf_pipe.extract(text):
        print(f"[{ent.label}] {ent.text} (conf: {ent.confidence:.2f})")

Исходный текст: Иван Петров купил акции Газпрома в Москве за 10000 рублей 5 мая 2023 года.

--- Natasha Extraction ---
[PER] Иван Петров (0-11)
[ORG] Газпрома (24-32)
[LOC] Москве (35-41)
[MONEY] 10000 рублей 5 (45-59)

--- Transformers Extraction (ruBert-tiny2) ---
Loading transformer model: Babelscape/wikineural-multilingual-ner...


Device set to use cpu


[PER] Иван Петров (conf: 1.00)
[ORG] Газпрома (conf: 0.96)
[LOC] Москве (conf: 1.00)


In [27]:
from dataclasses import dataclass
from typing import List, Set, Dict, Tuple, Any
from collections import defaultdict

# ==========================================
# 1. Обновленный Data Contract (Добавили eq/hash)
# ==========================================

@dataclass(frozen=True) # frozen=True делает объект неизменяемым и хешируемым
class Entity:
    text: str
    label: str
    start: int
    end: int
    confidence: float = 1.0

    # Метод для проверки пересечения спанов (для Partial matching)
    def intersects(self, other: 'Entity') -> bool:
        # Проверка пересечения отрезков [start, end]
        return (self.start < other.end) and (self.end > other.start)

# ==========================================
# 2. Класс для расчета метрик (SemEval style)
# ==========================================

@dataclass
class EvaluationMetrics:
    precision: float
    recall: float
    f1: float
    support: int  # Количество сущностей в эталоне

class NEREvaluator:
    """
    Класс для расчета метрик качества NER:
    - Strict Metrics (SemEval): точное совпадение границ и типа.
    - Partial Metrics: частичное пересечение границ + совпадение типа.
    """

    def evaluate(self, true_entities: List[Entity], pred_entities: List[Entity]) -> Dict[str, Any]:
        """
        Основной метод оценки.
        Возвращает словарь с метриками (общие + по классам).
        """
        # 1. Рассчитываем Strict Metrics (Строгие)
        strict_res = self._compute_metrics(true_entities, pred_entities, mode='strict')

        # 2. Рассчитываем Partial Metrics (Мягкие)
        partial_res = self._compute_metrics(true_entities, pred_entities, mode='partial')

        return {
            "strict": strict_res,
            "partial": partial_res
        }

    def _compute_metrics(self, y_true: List[Entity], y_pred: List[Entity], mode: str = 'strict') -> Dict[str, Any]:
        """
        Внутренняя логика подсчета TP, FP, FN.
        """
        # Счетчики: Global
        tp, fp, fn = 0, 0, 0

        # Счетчики: Per Class
        class_stats = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

        # Используем списки, чтобы помечать "использованные" сущности (чтобы не мапить одну сущность дважды)
        pred_matched = [False] * len(y_pred)
        true_matched = [False] * len(y_true)

        # 1. Ищем True Positives (TP)
        for i, pred in enumerate(y_pred):
            for j, true in enumerate(y_true):
                if true_matched[j]: continue # Эта сущность уже найдена

                is_match = False
                if mode == 'strict':
                    # Полное совпадение границ и лейбла
                    if pred.start == true.start and pred.end == true.end and pred.label == true.label:
                        is_match = True
                elif mode == 'partial':
                    # Пересечение границ и совпадение лейбла
                    if pred.intersects(true) and pred.label == true.label:
                        is_match = True

                if is_match:
                    tp += 1
                    class_stats[true.label]['tp'] += 1
                    pred_matched[i] = True
                    true_matched[j] = True
                    break # Переходим к следующему предикту

        # 2. Ищем False Positives (FP) - предсказали то, чего нет
        for i, matched in enumerate(pred_matched):
            if not matched:
                fp += 1
                class_stats[y_pred[i].label]['fp'] += 1

        # 3. Ищем False Negatives (FN) - пропустили то, что было
        for j, matched in enumerate(true_matched):
            if not matched:
                fn += 1
                class_stats[y_true[j].label]['fn'] += 1

        # Собираем итоговые метрики
        overall = self._calc_prf(tp, fp, fn)

        per_class = {}
        for label, stats in class_stats.items():
            per_class[label] = self._calc_prf(stats['tp'], stats['fp'], stats['fn'])

        return {
            "overall": overall,
            "per_class": per_class
        }

    def _calc_prf(self, tp: int, fp: int, fn: int) -> EvaluationMetrics:
        """Вспомогательная функция для формул"""
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        return EvaluationMetrics(
            precision=round(precision, 4),
            recall=round(recall, 4),
            f1=round(f1, 4),
            support=tp + fn
        )

# ==========================================
# 3. Пример использования (Integration Test)
# ==========================================

if __name__ == "__main__":
    # 1. Ground Truth (Эталон)
    # Текст: "Иван купил билет в Москву"
    true_entities = [
        Entity("Иван", "PER", 0, 4),
        Entity("Москву", "LOC", 19, 25)
    ]

    # 2. Prediction (Гипотеза модели)
    # Допустим, модель немного ошиблась в границах и нашла лишнее
    pred_entities = [
        Entity("Иван", "PER", 0, 4),           # Идеальное совпадение (Strict TP)
        Entity("в Москву", "LOC", 17, 25),    # Ошибка границ (Partial TP, Strict FP/FN)
        Entity("билет", "OBJ", 11, 16)        # Лишняя сущность (FP)
    ]

    evaluator = NEREvaluator()
    metrics = evaluator.evaluate(true_entities, pred_entities)

    print("=== Результаты оценки ===")

    print("\n[Strict Metrics] (Жесткие требования SemEval):")
    m = metrics['strict']['overall']
    print(f"Precision: {m.precision}, Recall: {m.recall}, F1: {m.f1}")

    print("\n[Partial Metrics] (Допускается пересечение границ):")
    m = metrics['partial']['overall']
    print(f"Precision: {m.precision}, Recall: {m.recall}, F1: {m.f1}")

    print("\n[Per Class Strict] (По классам):")
    for label, res in metrics['strict']['per_class'].items():
        print(f"Class {label}: F1 = {res.f1}")

=== Результаты оценки ===

[Strict Metrics] (Жесткие требования SemEval):
Precision: 0.3333, Recall: 0.5, F1: 0.4

[Partial Metrics] (Допускается пересечение границ):
Precision: 0.6667, Recall: 1.0, F1: 0.8

[Per Class Strict] (По классам):
Class PER: F1 = 1.0
Class LOC: F1 = 0.0
Class OBJ: F1 = 0.0


In [28]:
import os
from typing import List, Dict
from tqdm.auto import tqdm # Прогресс-бар

# Импортируем Natasha
from natasha import (
    Segmenter, MorphVocab, NewsEmbedding,
    NewsNERTagger, Doc
)

# ==========================================
# 1. Конфигурация Маппинга (NEREL -> Standard)
# ==========================================

# Превращаем детальные теги NEREL в общие, которые понимает Natasha
NEREL_TO_STD_MAP = {
    'PERSON': 'PER',
    'ORGANIZATION': 'ORG',
    'COUNTRY': 'LOC',
    'CITY': 'LOC',
    'STATE_OR_PROVINCE': 'LOC',
    'LOCATION': 'LOC',
    'FACILITY': 'LOC' # Иногда здания размечают как FACility, для Natasha это LOC/ORG
}

# ==========================================
# 2. Загрузчик данных (NEREL Parser)
# ==========================================

class NerelLoader:
    def __init__(self, dataset_path: str):
        self.dataset_path = dataset_path

    def load(self) -> List[Dict]:
        """
        Читает пары .txt и .ann, возвращает список словарей:
        [{'text': str, 'entities': List[Entity]}, ...]
        """
        documents = []

        # Получаем список файлов (убираем расширение)
        filenames = set()
        for f in os.listdir(self.dataset_path):
            if f.endswith('.txt'):
                filenames.add(f[:-4])

        print(f"Найдено документов: {len(filenames)}")

        for name in tqdm(filenames, desc="Loading NEREL"):
            txt_path = os.path.join(self.dataset_path, name + '.txt')
            ann_path = os.path.join(self.dataset_path, name + '.ann')

            # 1. Читаем текст
            with open(txt_path, 'r', encoding='utf-8') as f:
                text = f.read()

            # 2. Читаем аннотации
            entities = []
            if os.path.exists(ann_path):
                with open(ann_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        line = line.strip()
                        if not line.startswith('T'): continue # Нас интересуют только Text-bound entities

                        # Формат: T1  PERSON 10 20  Иван
                        parts = line.split('\t')
                        if len(parts) < 3: continue

                        tag_info = parts[1].split() # 'PERSON 10 20' или 'PERSON 10 15;16 20'
                        nerel_tag = tag_info[0]

                        # Парсим оффсеты (обрабатываем разрывные спаны, беря min и max)
                        # Пример: "10 15;16 20" -> start=10, end=20
                        spans = []
                        for s in tag_info[1:]:
                            if ';' in s:
                                sub_s = s.split(';')
                                spans.extend(sub_s)
                            else:
                                spans.append(s)

                        try:
                            start = int(spans[0])
                            end = int(spans[-1])
                        except ValueError:
                            continue # Пропускаем битую разметку

                        # Маппинг и фильтрация
                        # Если тег есть в нашем маппинге - берем, иначе пропускаем (т.к. Natasha его не знает)
                        if nerel_tag in NEREL_TO_STD_MAP:
                            mapped_tag = NEREL_TO_STD_MAP[nerel_tag]
                            entities.append(Entity(
                                text=parts[2],
                                label=mapped_tag,
                                start=start,
                                end=end
                            ))

            documents.append({
                "text": text,
                "entities": sorted(entities, key=lambda x: x.start)
            })

        return documents

# ==========================================
# 3. Реализация Natasha Pipeline (из прошлого шага)
# ==========================================

class NatashaBaseline:
    def __init__(self):
        self.segmenter = Segmenter()
        self.emb = NewsEmbedding()
        self.ner_tagger = NewsNERTagger(self.emb)
        self.morph_vocab = MorphVocab() # Для нормализации (опционально)

    def extract(self, text: str) -> List[Entity]:
        doc = Doc(text)
        doc.segment(self.segmenter)
        doc.tag_ner(self.ner_tagger)

        entities = []
        for span in doc.spans:
            # Natasha выдает PER, ORG, LOC - они уже соответствуют нашему стандарту
            entities.append(Entity(
                text=span.text,
                label=span.type,
                start=span.start,
                end=span.stop
            ))
        return entities

# ==========================================
# 4. Основной скрипт выполнения
# ==========================================

def run_baseline_evaluation():
    # Путь к данным (из твоего запроса)
    DATA_PATH = "/content/NEREL/NEREL-v1.1/test"

    # 1. Загрузка
    loader = NerelLoader(DATA_PATH)
    data = loader.load()

    if not data:
        print("Ошибка: Данные не загружены. Проверь путь.")
        return

    # 2. Инициализация модели
    print("Инициализация Natasha...")
    model = NatashaBaseline()

    # 3. Инициализация оценщика (класс из прошлого ответа)
    evaluator = NEREvaluator()

    # 4. Прогон по всему датасету
    all_true = []
    all_pred = []

    print("Запуск инференса...")
    for doc in tqdm(data, desc="Processing"):
        text = doc['text']
        true_ents = doc['entities']

        # Предсказание
        pred_ents = model.extract(text)

        all_true.extend(true_ents)
        all_pred.extend(pred_ents)

    # 5. Подсчет метрик
    print("\n" + "="*30)
    print("РЕЗУЛЬТАТЫ NATASHA (BASELINE)")
    print("="*30)

    results = evaluator.evaluate(all_true, all_pred)

    # Вывод
    def print_metrics(metrics_dict, title):
        print(f"\n--- {title} ---")
        ov = metrics_dict['overall']
        print(f"[OVERALL] Precision: {ov.precision:.2f} | Recall: {ov.recall:.2f} | F1: {ov.f1:.2f}")
        print("По классам:")
        for label, m in metrics_dict['per_class'].items():
            print(f"  {label:<5}: P={m.precision:.2f} R={m.recall:.2f} F1={m.f1:.2f} (Supp: {m.support})")

    print_metrics(results['strict'], "Strict Match (Строгое совпадение)")
    print_metrics(results['partial'], "Partial Match (Частичное совпадение)")

# Запуск
if __name__ == "__main__":
    # Убедитесь, что Entity и NEREvaluator определены в ноутбуке!
    try:
        run_baseline_evaluation()
    except FileNotFoundError:
        print("Папка с данными не найдена. Проверьте путь.")

Найдено документов: 93


Loading NEREL:   0%|          | 0/93 [00:00<?, ?it/s]

Инициализация Natasha...
Запуск инференса...


Processing:   0%|          | 0/93 [00:00<?, ?it/s]


РЕЗУЛЬТАТЫ NATASHA (BASELINE)

--- Strict Match (Строгое совпадение) ---
[OVERALL] Precision: 0.87 | Recall: 0.71 | F1: 0.78
По классам:
  LOC  : P=0.93 R=0.69 F1=0.80 (Supp: 932)
  ORG  : P=0.72 R=0.57 F1=0.64 (Supp: 675)
  PER  : P=0.90 R=0.84 F1=0.87 (Supp: 961)

--- Partial Match (Частичное совпадение) ---
[OVERALL] Precision: 0.99 | Recall: 0.81 | F1: 0.89
По классам:
  LOC  : P=0.99 R=0.74 F1=0.85 (Supp: 932)
  ORG  : P=0.97 R=0.76 F1=0.85 (Supp: 675)
  PER  : P=0.99 R=0.92 F1=0.95 (Supp: 961)


In [34]:
# @title 7. Исправленная реализация DeepPavlov (RuBERT Collection3)
from transformers import pipeline
import torch
import numpy as np

class DeepPavlovNER(BaseNERPipeline):
    def __init__(self, device: int = 0):
        # Используем модель, обученную на Collection3 (стандарт для русского NER)
        # Это наиболее близкий аналог моделей DeepPavlov
        self.model_name = "viktoroo/sberbank-rubert-base-collection3"
        print(f"Загрузка модели {self.model_name}...")

        try:
            self.pipeline = pipeline(
                "ner",
                model=self.model_name,
                tokenizer=self.model_name,
                aggregation_strategy="simple", # Склеивает токены
                device=device
            )
        except Exception as e:
            print(f"Ошибка загрузки: {e}")
            self.pipeline = None

        # МАППИНГ ТЕГОВ (Самая важная часть исправления)
        # Приводим всё многообразие тегов к трем стандартным
        self.tag_mapping = {
            'PER': 'PER', 'PERSON': 'PER', 'B-PER': 'PER', 'I-PER': 'PER',
            'ORG': 'ORG', 'ORGANIZATION': 'ORG', 'B-ORG': 'ORG', 'I-ORG': 'ORG',
            'LOC': 'LOC', 'LOCATION': 'LOC', 'B-LOC': 'LOC', 'I-LOC': 'LOC',
            'GPE': 'LOC', 'GP': 'LOC' # Гео-политические сущности туда же
        }

    def extract(self, text: str) -> list:
        if not self.pipeline: return []

        # SLIDING WINDOW (Скользящее окно)
        # BERT падает или обрезает тексты длиннее 512 токенов.
        # Мы бьем текст на куски, прогоняем и склеиваем результаты.

        chunk_size = 1000  # Символов (примерно 200-300 токенов)
        overlap = 100      # Перекрытие, чтобы не разрезать сущность посередине

        entities = []
        text_len = len(text)

        # Если текст короткий, обрабатываем целиком
        if text_len <= chunk_size:
            chunks = [(0, text)]
        else:
            chunks = []
            start = 0
            while start < text_len:
                end = min(start + chunk_size, text_len)
                chunk_text = text[start:end]
                chunks.append((start, chunk_text))
                if end == text_len: break
                start += chunk_size - overlap # Сдвигаем окно

        # Прогон по кускам
        for offset, chunk_text in chunks:
            try:
                # Инференс
                results = self.pipeline(chunk_text)

                for item in results:
                    # Получаем тег
                    raw_label = item['entity_group']

                    # Маппинг (Исправление ошибки 0.00)
                    label = self.tag_mapping.get(raw_label)

                    # Если тег не в нашем списке (например, MISC), пропускаем
                    if label:
                        entities.append(Entity(
                            text=item['word'],
                            label=label,
                            start=offset + item['start'], # Adjust start position for chunk
                            end=offset + item['end'],     # Adjust end position for chunk
                            confidence=item['score']
                        ))
            except Exception as e:
                print(f"Error during chunk inference: {e}")
                continue # Continue to the next chunk even if one fails

        # Sort entities by their start position after combining all chunks
        return sorted(entities, key=lambda x: x.start)


In [35]:
def run_improved_comparison():
    # 1. Настройки
    DATA_PATH = "/content/NEREL/NEREL-v1.1/test"
    LIMIT_DOCS = 20  # Меньше документов для быстрого теста

    # 2. Загрузка
    print(">>> Загрузка данных...")
    loader = NerelLoader(DATA_PATH)
    data = loader.load()
    if not data: return
    test_data = data[:LIMIT_DOCS]

    # 3. Инициализация
    print("\n>>> Инициализация модели DeepPavlov (Collection3)...")
    device = 0 if torch.cuda.is_available() else -1
    dp_model = DeepPavlovNER(device=device)

    # ТЕСТОВЫЙ ПРОГОН НА ОДНОМ ПРИМЕРЕ (DEBUG)
    print("\n--- DEBUG: Проверка на одном примере ---")
    sample_text = test_data[0]['text'][:300]
    print(f"Текст: {sample_text}...")
    debug_preds = dp_model.extract(sample_text)
    print("Найденные сущности:", [(e.text, e.label) for e in debug_preds])
    print("----------------------------------------\n")

    # 4. Основной цикл
    evaluator = NEREvaluator()
    all_true = []
    dp_preds = []

    print(f">>> Обработка {len(test_data)} документов...")
    for doc in tqdm(test_data, desc="Inference"):
        all_true.extend(doc['entities'])
        dp_preds.extend(dp_model.extract(doc['text']))

    # 5. Результаты
    print("\n" + "="*40)
    print(" РЕЗУЛЬТАТЫ (RuBERT Collection3) ")
    print("="*40)

    res = evaluator.evaluate(all_true, dp_preds)

    # Strict
    s = res['strict']['overall']
    print(f"[Strict Match]  F1: {s.f1:.2f} | P: {s.precision:.2f} | R: {s.recall:.2f}")

    # Partial
    p = res['partial']['overall']
    print(f"[Partial Match] F1: {p.f1:.2f} (С учетом ошибок границ)")

    print("\nПо классам (Strict F1):")
    for tag, metrics in res['strict']['per_class'].items():
        print(f"  {tag:<4}: {metrics.f1:.2f} (Support: {metrics.support})")

# Запуск
if __name__ == "__main__":
    try:
        run_improved_comparison()
    except Exception as e:
        print(f"Ошибка: {e}")

>>> Загрузка данных...
Найдено документов: 93


Loading NEREL:   0%|          | 0/93 [00:00<?, ?it/s]


>>> Инициализация модели DeepPavlov (Collection3)...
Загрузка модели viktoroo/sberbank-rubert-base-collection3...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

...

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



--- DEBUG: Проверка на одном примере ---
Текст: Умер раненный в Афганистане американский военный
Американские военные в провинции Бадгис, Афганистан, 2011 год
Американский военный умер от ранений, полученных во время боя в Афганистане. Об этом 18 января сообщило Министерство обороны США.

26-летний сержант Кэмерон Мэддок (Cameron A. Meddock) из С...
Найденные сущности: [('афганистане', 'LOC'), ('бадгис', 'LOC'), ('афганистан', 'LOC'), ('афганистане', 'LOC'), ('министерство обороны', 'ORG'), ('сша', 'LOC'), ('кэмерон мэддок ( cameron a. meddock )', 'PER')]
----------------------------------------

>>> Обработка 20 документов...


Inference:   0%|          | 0/20 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



 РЕЗУЛЬТАТЫ (RuBERT Collection3) 
[Strict Match]  F1: 0.75 | P: 0.78 | R: 0.73
[Partial Match] F1: 0.82 (С учетом ошибок границ)

По классам (Strict F1):
  LOC : 0.75 (Support: 190)
  ORG : 0.58 (Support: 141)
  PER : 0.86 (Support: 207)


In [1]:
API_KEY = "sk-kDGfybeelE64OS3aSPydFw"

In [2]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install openai rich
!pip install httpx

downloading uv 0.9.16 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.12.12 environment at: /usr
Audited 2 packages in 110ms


In [3]:
from openai import OpenAI
from rich import print, inspect
from pydantic import BaseModel
import httpx
%load_ext rich

In [4]:
# @title старт прогона
def get_spending_status():
    _spending_check_key = "sk-9XRTyzxzBT66meBLqdgKZw" # restricted, can't be used for other api requests
    spend_response = httpx.get(
        "https://llm.buffedby.ai/key/info",
        params={
            'key': API_KEY
        },
        headers={
            'Authorization': f"Bearer {_spending_check_key}"
        }
    ).json()
    budget = spend_response['info']['max_budget']
    spend = spend_response['info']['spend']
    remaining = budget - spend
    print(f"Key Alias: {spend_response['info']['key_alias']}")
    print(f"Spending Limit: {budget:.4f}")
    print(f"Spend:          {spend:.4f}")
    print(f"Remaining:      {remaining:.4f}")

get_spending_status()

Key Alias: RUDN-Key-4

Spending Limit: 30.0000

Spend:          2.4128

Remaining:      27.5872

In [5]:
# @title четверть фунтовый
def get_spending_status():
    _spending_check_key = "sk-9XRTyzxzBT66meBLqdgKZw" # restricted, can't be used for other api requests
    spend_response = httpx.get(
        "https://llm.buffedby.ai/key/info",
        params={
            'key': API_KEY
        },
        headers={
            'Authorization': f"Bearer {_spending_check_key}"
        }
    ).json()
    budget = spend_response['info']['max_budget']
    spend = spend_response['info']['spend']
    remaining = budget - spend
    print(f"Key Alias: {spend_response['info']['key_alias']}")
    print(f"Spending Limit: {budget:.4f}")
    print(f"Spend:          {spend:.4f}")
    print(f"Remaining:      {remaining:.4f}")

get_spending_status()

Key Alias: RUDN-Key-4

Spending Limit: 30.0000

Spend:          3.0125

Remaining:      26.9875

In [6]:
# @title zero shote
def get_spending_status():
    _spending_check_key = "sk-9XRTyzxzBT66meBLqdgKZw" # restricted, can't be used for other api requests
    spend_response = httpx.get(
        "https://llm.buffedby.ai/key/info",
        params={
            'key': API_KEY
        },
        headers={
            'Authorization': f"Bearer {_spending_check_key}"
        }
    ).json()
    budget = spend_response['info']['max_budget']
    spend = spend_response['info']['spend']
    remaining = budget - spend
    print(f"Key Alias: {spend_response['info']['key_alias']}")
    print(f"Spending Limit: {budget:.4f}")
    print(f"Spend:          {spend:.4f}")
    print(f"Remaining:      {remaining:.4f}")

get_spending_status()

Key Alias: RUDN-Key-4

Spending Limit: 30.0000

Spend:          3.8007

Remaining:      26.1993

In [7]:
# @title fin
def get_spending_status():
    _spending_check_key = "sk-9XRTyzxzBT66meBLqdgKZw" # restricted, can't be used for other api requests
    spend_response = httpx.get(
        "https://llm.buffedby.ai/key/info",
        params={
            'key': API_KEY
        },
        headers={
            'Authorization': f"Bearer {_spending_check_key}"
        }
    ).json()
    budget = spend_response['info']['max_budget']
    spend = spend_response['info']['spend']
    remaining = budget - spend
    print(f"Key Alias: {spend_response['info']['key_alias']}")
    print(f"Spending Limit: {budget:.4f}")
    print(f"Spend:          {spend:.4f}")
    print(f"Remaining:      {remaining:.4f}")

get_spending_status()

Key Alias: RUDN-Key-4

Spending Limit: 30.0000

Spend:          5.4199

Remaining:      24.5801

In [53]:
import json
import openai
import re
from tqdm.auto import tqdm
import pandas as pd

# ==========================================
# 0. УБЕДИМСЯ, ЧТО БАЗОВЫЕ КЛАССЫ ОПРЕДЕЛЕНЫ
# (Если вы выполняли ячейки выше, они уже есть,
# но для надежности напомним структуру)
# ==========================================

from dataclasses import dataclass
from typing import List

# Если класс Entity еще не определен в памяти ноутбука:
@dataclass(frozen=True)
class Entity:
    text: str
    label: str
    start: int
    end: int
    confidence: float = 1.0

    def intersects(self, other: 'Entity') -> bool:
        return (self.start < other.end) and (self.end > other.start)

# ==========================================
# 1. ПОДГОТОВКА ДАННЫХ
# ==========================================

# Загружаем данные
try:
    with open('dataset.json', 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
except FileNotFoundError:
    print("Файл dataset.json не найден! Выполните код конвертации.")
    raw_data = []

# Делим на примеры (few-shot) и тест
# data_dict: { "Текст новости...": {"Кто": ["Apple"], "Регион": ["США"]} }
items = [(d['text'], d['labels']) for d in raw_data]

# Берем 3 примера для few-shot и 20 для теста
examples_pool = items[:3]
validation_data = items[3:23]

# ==========================================
# 2. НАСТРОЙКА LLM (RUDN Proxy / OpenAI)
# ==========================================


MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"

client = openai.OpenAI(
    api_key=API_KEY,
    base_url="https://llm.buffedby.ai/v1"
)

# ==========================================
# 3. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ==========================================

def clean_json_response(response_text):
    """Вырезает JSON из Markdown разметки"""
    match = re.search(r'\{.*\}', response_text, re.DOTALL)
    json_str = match.group(0) if match else response_text
    try:
        return json.loads(json_str)
    except:
        return {}

def json_to_entities(text: str, json_data: dict) -> List[Entity]:
    """
    Превращает словарь {"Кто": ["Apple"]} в список объектов Entity.
    Ищет подстроки в тексте, чтобы найти start/end.
    """
    entities = []
    if not isinstance(json_data, dict):
        return []

    for label, values in json_data.items():
        if not isinstance(values, list): continue

        for val in values:
            if not isinstance(val, str): continue

            # Чистим значение от лишних пробелов
            val = val.strip()
            if not val: continue

            # Ищем первое вхождение в тексте
            # (Для продакшена можно искать все вхождения, но для теста хватит первого)
            start = text.find(val)
            if start != -1:
                end = start + len(val)
                entities.append(Entity(
                    text=val,
                    label=label, # "Кто", "Регион" и т.д.
                    start=start,
                    end=end
                ))
    return entities

def extract_with_llama(text, few_shot_examples=None):
    """Отправляет запрос к LLM"""
    system_prompt = (
        "Ты — эксперт NER. Извлеки сущности из текста в формате JSON.\n"
        "Поля: 'Кто', 'Тип вложения', 'Цель', 'Регион', 'Стоимость', 'Дата'.\n"
        "Если не найдено — пустой список []. Только JSON."
    )

    messages = [{"role": "system", "content": system_prompt}]

    # Добавляем примеры
    if few_shot_examples:
        for ex_text, ex_labels in few_shot_examples:
            messages.append({"role": "user", "content": ex_text})
            messages.append({"role": "assistant", "content": json.dumps(ex_labels, ensure_ascii=False)})

    messages.append({"role": "user", "content": text})

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.0,
            max_tokens=512
        )
        return clean_json_response(response.choices[0].message.content)
    except Exception as e:
        print(f"Error: {e}")
        return {}

# ==========================================
# 4. ОСНОВНОЙ ЭКСПЕРИМЕНТ (С использованием NEREvaluator)
# ==========================================

# Предполагаем, что класс NEREvaluator определен выше (шаг 6).
# Если нет - скопируйте его код сюда.
try:
    evaluator = NEREvaluator()
except NameError:
    print("ОШИБКА: Класс NEREvaluator не найден. Пожалуйста, выполните ячейку с его кодом (шаг 6).")
    # Заглушка, чтобы код не падал, если вы забыли
    class NEREvaluator:
        def evaluate(self, t, p): return {"strict": {"overall": type('obj', (object,), {'f1': 0.0})}}
    evaluator = NEREvaluator()

results_summary = []

# Тестируем разные режимы
shots_scenarios = [0, 1, 5] if len(examples_pool) >= 1 else [0]

for shots in shots_scenarios:
    print(f"\n--- Запуск Llama-3.3 ({shots}-shot) ---")

    current_examples = examples_pool[:shots] if shots > 0 else None

    all_true_entities = []
    all_pred_entities = []

    for text, true_labels_dict in tqdm(validation_data, desc="Inference"):
        # 1. Получаем предсказание (JSON)
        pred_json = extract_with_llama(text, current_examples)

        # 2. Конвертируем JSON -> Entity (восстанавливаем координаты)
        pred_ents = json_to_entities(text, pred_json)
        true_ents = json_to_entities(text, true_labels_dict)

        # 3. Собираем для общей оценки
        all_true_entities.extend(true_ents)
        all_pred_entities.extend(pred_ents)

    # 4. Считаем метрики НАШИМ классом
    metrics = evaluator.evaluate(all_true_entities, all_pred_entities)

    # 5. Вывод
    strict = metrics['strict']['overall']
    partial = metrics['partial']['overall']

    print(f"Результат {shots}-shot:")
    print(f"  Strict F1:  {strict.f1:.2f} (P: {strict.precision:.2f}, R: {strict.recall:.2f})")
    print(f"  Partial F1: {partial.f1:.2f}")

    results_summary.append({
        "Shots": shots,
        "Strict F1": strict.f1,
        "Partial F1": partial.f1,
        "Precision": strict.precision,
        "Recall": strict.recall
    })

# Итоговая таблица
print("\n=== ИТОГОВАЯ ТАБЛИЦА (Llama-3.3) ===")
df_res = pd.DataFrame(results_summary)
display(df_res)

--- Запуск Llama-3.3 (0-shot) ---

Inference:   0%|          | 0/20 [00:00<?, ?it/s]

Результат 0-shot:

Strict F1:  0.28 (P: 0.72, R: 0.17)

Partial F1: 0.37

...

Результат 5-shot:

Strict F1:  0.47 (P: 0.73, R: 0.34)

Partial F1: 0.61

=== ИТОГОВАЯ ТАБЛИЦА (Llama-3.3) ===

,Shots,Strict F1,Partial F1,Precision,Recall
0,0,0.2791,0.3714,0.7175,0.1733
1,1,0.3539,0.4874,0.6836,0.2387
2,5,0.4657,0.6104,0.7275,0.3424
